In [8]:
# =========================================================
# XGBOOST-LIKE CLASSIFIER (OPTIMIZED WITH SKLEARN)
# PREDICTING: LABEL + CONFIDENT
# =========================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# =========================================================
# 1. LOAD DATA
# =========================================================

# Đảm bảo đường dẫn file chính xác
df = pd.read_csv("/home/nguyenhuynh/Documents/appcuuho/be/data/dataset/data_v5.csv") # Cập nhật đường dẫn của bạn nếu cần

# =========================================================
# 2. DATA CLEANING & TARGET PREPARATION
# =========================================================

# Chuẩn hóa nhãn (tránh lỗi do viết hoa/thường)
df['label'] = df['label'].astype(str).str.strip().str.upper()
df['confident'] = df['confident'].astype(str).str.strip().str.upper()

# Mã hóa Label (Mức độ khẩn cấp)
le_label = LabelEncoder()
y_label = le_label.fit_transform(df['label'])
print("Labels (Emergency):", le_label.classes_)

# Mã hóa Confident (Độ tin cậy)
le_conf = LabelEncoder()
y_conf = le_conf.fit_transform(df['confident'])
print("Labels (Confident):", le_conf.classes_)

# =========================================================
# 3. FEATURE ENGINEERING
# =========================================================

# Tạo các đặc trưng tính toán (giữ lại logic của bạn)
df['humanitarian_need_score'] = (
    df['lương thực & nước uống'] +
    df['thuốc men & hỗ trợ y tế'] +
    df['chỗ ở & nơi trú ẩn']
)

df['rescue_need_score'] = (
    df['phương tiện & cứu hộ'] +
    df['vật dụng sinh hoạt (áo phao, đèn pin, quần áo, mền chăn)']
)

df['affected_people_log'] = np.log1p(df['affected_people'])

# Chọn các đặc trưng số (Numerical features)
num_features = [
    'disaster_type',
    'affected_people',
    'deathly',
    'is_isolated',
    'infrastructure_damage',
    'lương thực & nước uống',
    'thuốc men & hỗ trợ y tế',
    'phương tiện & cứu hộ',
    'chỗ ở & nơi trú ẩn',
    'vật dụng sinh hoạt (áo phao, đèn pin, quần áo, mền chăn)',
    'humanitarian_need_score',
    'rescue_need_score',
    'affected_people_log'
]

# Chọn đặc trưng phân loại (Categorical features)
# Thay vì dùng 'location' (quá chi tiết), ta dùng 'tinh_tp' (Tỉnh/TP) để tổng quát hóa tốt hơn
cat_features = ['tinh_tp']

X = df[num_features + cat_features].copy()

# Xử lý dữ liệu (Imputation + Encoding)
# Dùng ColumnTransformer để xử lý riêng số và chữ
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='constant', fill_value=0), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ]
)

# =========================================================
# 4. TRAIN / TEST SPLIT (FIXED RANDOM STATE)
# =========================================================

# Chia tập train/test cố định với random_state=42
# Stratify theo y_label để đảm bảo tỷ lệ nhãn cân bằng
X_train, X_test, y_label_train, y_label_test, y_conf_train, y_conf_test = train_test_split(
    X, y_label, y_conf, test_size=0.1, stratify=y_label, random_state=42
)

# =========================================================
# 5. MODEL DEFINITION (GRADIENT BOOSTING - "XGBoost-like")
# =========================================================

# Sử dụng GradientBoostingClassifier từ sklearn (mạnh mẽ và ổn định hơn code tự viết)
# Đây chính là lõi thuật toán giống XGBoost
def create_model():
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', GradientBoostingClassifier(
            n_estimators=200,      # Số lượng cây
            learning_rate=0.05,    # Tốc độ học
            max_depth=4,           # Độ sâu cây
            random_state=42        # CỐ ĐỊNH RANDOM STATE ĐỂ KẾT QUẢ KHÔNG ĐỔI
        ))
    ])

# =========================================================
# 6. TRAINING & EVALUATION
# =========================================================

print("\n" + "="*40)
print("TRAINING MODEL 1: EMERGENCY LABEL")
print("="*40)

model_label = create_model()
model_label.fit(X_train, y_label_train)

# Dự đoán trên tập Test
y_pred_label = model_label.predict(X_test)
acc_label = accuracy_score(y_label_test, y_pred_label)

print(f"Test Accuracy (Label): {acc_label:.4f}")
print("\nClassification Report (Label):")
print(classification_report(y_label_test, y_pred_label, target_names=le_label.classes_))

# ---------------------------------------------------------

print("\n" + "="*40)
print("TRAINING MODEL 2: CONFIDENT SCORE")
print("="*40)

model_conf = create_model()
model_conf.fit(X_train, y_conf_train)

# Dự đoán trên tập Test
y_pred_conf = model_conf.predict(X_test)
acc_conf = accuracy_score(y_conf_test, y_pred_conf)

print(f"Test Accuracy (Confident): {acc_conf:.4f}")
print("\nClassification Report (Confident):")
print(classification_report(y_conf_test, y_pred_conf, target_names=le_conf.classes_))

# =========================================================
# 7. CROSS VALIDATION (CHECK STABILITY)
# =========================================================

print("\n" + "="*40)
print("STABILITY CHECK (CROSS VALIDATION)")
print("="*40)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_label, X, y_label, cv=cv, scoring='accuracy')

print(f"Cross-val Accuracy Scores: {scores}")
print(f"Mean Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

Labels (Emergency): ['CAO' 'THẤP' 'TRUNG BÌNH']
Labels (Confident): ['CAO' 'THẤP']

TRAINING MODEL 1: EMERGENCY LABEL
Test Accuracy (Label): 0.4694

Classification Report (Label):
              precision    recall  f1-score   support

         CAO       0.70      0.62      0.65        26
        THẤP       0.27      0.43      0.33         7
  TRUNG BÌNH       0.27      0.25      0.26        16

    accuracy                           0.47        49
   macro avg       0.41      0.43      0.41        49
weighted avg       0.50      0.47      0.48        49


TRAINING MODEL 2: CONFIDENT SCORE
Test Accuracy (Confident): 0.9592

Classification Report (Confident):
              precision    recall  f1-score   support

         CAO       0.98      0.98      0.98        44
        THẤP       0.80      0.80      0.80         5

    accuracy                           0.96        49
   macro avg       0.89      0.89      0.89        49
weighted avg       0.96      0.96      0.96        49


STABIL

In [ ]:
import pandas as pd
from models import Train_model_XGBoost 

# ==============================================================================
# 1. TẠO DỮ LIỆU TEST MẪU (Giả lập dữ liệu từ API/Frontend gửi xuống)
# ==============================================================================
test_data = [
    # CASE 1: Mức độ CAO (Lũ lụt, nhiều người bị kẹt, có người già/trẻ em, thiếu lương thực)
    {
        'content': 'Cứu gấp, ngõ 770 Dương Tự Minh, 35 người kẹt, có trẻ 6 tháng, nước ngập tầng 2, không thức ăn',
        'disaster_type': 2,           # Lũ lụt
        'affected_people': 35,        # Rất đông người
        'deathly': 0,                 # Chưa có người chết (nhưng nguy hiểm)
        'is_isolated': 1,             # Bị cô lập
        'infrastructure_damage': 1,   # Nhà cửa hư hại
        'lương thực & nước uống': 1,  # Cần gấp
        'phương tiện & cứu hộ': 1,    # Cần xuồng
        'tinh_tp': 'Thái Nguyên'      # Địa điểm
    },
    
    # CASE 2: Mức độ TRUNG BÌNH (Ít người, chưa nguy hiểm tính mạng ngay lập tức)
    {
        'content': 'Khu vực Đồng Hỷ, nhà mình 3 người lớn bị kẹt, hiện tại vẫn ổn nhưng hết đồ ăn từ hôm qua',
        'disaster_type': 2,
        'affected_people': 3,
        'deathly': 0,
        'is_isolated': 1,
        'infrastructure_damage': 0,
        'lương thực & nước uống': 1,  # Chỉ cần lương thực
        'phương tiện & cứu hộ': 0,    # Chưa cần cứu hộ gấp
        'tinh_tp': 'Thái Nguyên'
    },

    # CASE 3: Mức độ THẤP (Thông báo tin tức hoặc xin hỗ trợ vật dụng sau bão)
    {
        'content': 'Cần xin ít quần áo cũ và chăn màn cho bà con vùng lũ đã rút nước',
        'disaster_type': 0,
        'affected_people': 0,
        'deathly': 0,
        'is_isolated': 0,             # Không bị cô lập
        'infrastructure_damage': 0,
        'lương thực & nước uống': 0,
        'vật dụng sinh hoạt (áo phao, đèn pin, quần áo, mền chăn)': 1, # Cần vật dụng
        'tinh_tp': 'Hà Nội'
    }
]

# Chuyển list dictionary thành DataFrame để đưa vào model
df_test = pd.DataFrame(test_data)

# ==============================================================================
# 2. HUẤN LUYỆN & DỰ ĐOÁN
# ==============================================================================

# Bước A: Load dữ liệu gốc để train (Học lại từ đầu)
# (Trong thực tế bạn nên train 1 lần rồi lưu model lại, nhưng ở đây mình train luôn để demo)
print("⏳ Đang huấn luyện mô hình...")
df_train = pd.read_csv("/home/nguyenhuynh/Documents/appcuuho/be/data/dataset/data_v5.csv") 
brain = Train_model_XGBoost.train(df_train) # Train xong có bộ não "brain"

print("\n" + "="*50)
print("KẾT QUẢ DỰ ĐOÁN TRÊN TẬP TEST MẪU")
print("="*50)

# Bước B: Chạy dự đoán từng dòng
for i, row in df_test.iterrows():
    # Gọi hàm dự đoán
    result = Train_model_XGBoost.predict(brain, row.to_dict())
    
    # In kết quả đẹp
    print(f"\n🔹 Trường hợp {i+1}: {row['content'][:60]}...")
    print(f"   -> Mức độ khẩn cấp: {result['urgency']} (Độ chắc chắn: {result['urgency_prob']:.2f})")
    print(f"   -> Độ tin cậy tin:  {result['confidence']} (Độ chắc chắn: {result['confidence_prob']:.2f})")
    
    # Logic xử lý thêm (Ví dụ cho Backend)
    if result['urgency'] == 'CAO':
        print("   ⚠️  CẢNH BÁO: Cần gửi đội cứu hộ ngay lập tức!")

⏳ Đang huấn luyện mô hình...
🔄 Đang huấn luyện mô hình... (Config: n=200, lr=0.05)
✅ Đã lưu model thành công tại: rescue_model.pkl

KẾT QUẢ DỰ ĐOÁN TRÊN TẬP TEST MẪU

🔹 Trường hợp 1: Cứu gấp, ngõ 770 Dương Tự Minh, 35 người kẹt, có trẻ 6 tháng...
   -> Mức độ khẩn cấp: CAO (Độ chắc chắn: 0.69)
   -> Độ tin cậy tin:  CAO (Độ chắc chắn: 1.00)
   ⚠️  CẢNH BÁO: Cần gửi đội cứu hộ ngay lập tức!

🔹 Trường hợp 2: Khu vực Đồng Hỷ, nhà mình 3 người lớn bị kẹt, hiện tại vẫn ổ...
   -> Mức độ khẩn cấp: TRUNG BÌNH (Độ chắc chắn: 0.45)
   -> Độ tin cậy tin:  CAO (Độ chắc chắn: 0.99)

🔹 Trường hợp 3: Cần xin ít quần áo cũ và chăn màn cho bà con vùng lũ đã rút ...
   -> Mức độ khẩn cấp: THẤP (Độ chắc chắn: 0.68)
   -> Độ tin cậy tin:  CAO (Độ chắc chắn: 0.97)
